In [ ]:
!pip install -q transformers==4.45.2
!pip install -q tokenizers==0.20.1
!pip install -q sentencepiece==0.2.0

In [ ]:
!pip install -q arabert

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 14.7 MB/s eta 0:00:00


In [ ]:
!pip install -q sacrebleu
!pip install -q rouge-score
!pip install -q bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 8.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.9 MB/s eta 0:00:00


In [ ]:
!pip install -U evaluate==0.4.6 huggingface_hub

# Summarization

## Load dataset

In [ ]:
import json, re
from html import unescape
from pathlib import Path
from typing import Union, Tuple, Dict
import numpy as np
import pandas as pd
def read_jsonl(path: Union[str, Path]) -> pd.DataFrame:
    path = Path(path)
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for ln in f:
            ln = ln.strip()
            if ln:
                rows.append(json.loads(ln))
    return pd.DataFrame(rows)

In [ ]:
train = read_jsonl("arabic_train.jsonl")
train['split'] = 'train'

dev   = read_jsonl("arabic_val.jsonl")
dev['split'] = 'dev'

test  = read_jsonl("arabic_test.jsonl")
test['split'] = 'test'

In [ ]:
train.shape[0], dev.shape[0], test.shape[0]

(37519, 4689, 4689)

In [ ]:
sum = pd.concat([train, dev, test])
sum = sum[['text', 'summary', 'split']]
sum.head(2)

,text,summary,split
0,وكان الرئيس الأوكراني المؤقت، الكسندر تورتشينو...,بدأت القوات الأوكرانية الانسحاب من شبه جزيرة ا...,train
1,بحلول عام 2050 ستحتاج مصر إلى 21 مليار متر مكع...,"""هل سيتم تغيير العبارة الشهيرة للمؤرخ اليوناني...",train


In [ ]:
sum.shape[0]

46897

In [ ]:
test_samples = pd.read_excel('Samples Test.xlsx')
test_samples = test_samples.rename(columns = {'Text':'text'})
test_samples.head(1)

,text,summary
0,واستحوذت القصة على نقاشات رواد مواقع التواصل ا...,أثار مقطع فيديو يظهر طفلا يقود سيارة ويعتدي لف...


In [ ]:
test_samples.shape[0]

200

In [ ]:
sum_filtered = sum[~sum["text"].isin(test_samples["text"])]
sum_filtered .shape[0]

46697

# Split Data

In [ ]:
from datasets import Dataset, DatasetDict

# Create datasets for each split
billsum = DatasetDict({
    split: Dataset.from_pandas(
        sum_filtered[sum_filtered["split"] == split][["text", "summary"]],
        preserve_index=False
    )
    for split in ["train", "dev", "test"]
})

## Preprocess

In [ ]:
from transformers import AutoTokenizer

checkpoint = "UBC-NLP/AraT5v2-base-1024"
tokenizer = AutoTokenizer.from_pretrained(
    checkpoint,
    use_fast=False
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [ ]:
prefix = "لخص: "


def preprocess_function(examples):
    inputs = [prefix + doc for doc in examples["text"]]
    model_inputs = tokenizer(inputs, max_length=1024, truncation=True)

    labels = tokenizer(text_target=examples["summary"], max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
tokenized_billsum = billsum.map(preprocess_function, batched=True)

Map:   0%|          | 0/37519 [00:00<?, ? examples/s]

Map:   0%|          | 0/4689 [00:00<?, ? examples/s]

Map:   0%|          | 0/4489 [00:00<?, ? examples/s]

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=checkpoint)

## Evaluate

In [ ]:
import evaluate

rouge = evaluate.load("rouge")

In [ ]:
import numpy as np


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

## Train

In [ ]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="my_awesome_billsum_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=4,
    predict_with_generate=True,
    fp16=True,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_billsum["train"],
    eval_dataset=tokenized_billsum["dev"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,3.301100,2.429738,0.020600,0.001400,0.020900,0.020800,18.820200
2,3.090900,2.377297,0.022400,0.001600,0.022500,0.022700,18.800400
3,3.000300,2.353114,0.023300,0.001800,0.023400,0.023500,18.796500
4,2.945900,2.342810,0.025000,0.002500,0.025100,0.025200,18.767800


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1220: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1220: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1220: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1220: UserWarning: Using the model-agnostic default `max_length` (=20) to control

TrainOutput(global_step=9380, training_loss=3.201646803996202, metrics={'train_runtime': 4149.8293, 'train_samples_per_second': 36.164, 'train_steps_per_second': 2.26, 'total_flos': 2.563512243485737e+17, 'train_loss': 3.201646803996202, 'epoch': 4.0})

## Inference

In [ ]:
text = 'واستحوذت القصة على نقاشات رواد مواقع التواصل الاجتماعي، حيث تصدرت وسوم مثل "#ابن_القاضي وشرطي المرور" قائمة المواضيع المتداولة على تويتر لساعات طويلة. واكب المغردون تطورات القضية بدءا من مرحلة استجواب الطفل والإفراج عنه وحتى إعادة توقيف وإيداعه لدار رعاية في وقت لاحق. فقد أفادت تقارير صحفية بإعادة توقيف الطفل وأربعة من رفاقه صباح الاثنين 2 نوفمبر/ تشرين الثاني. ووصفت تلك التقارير حالة "الهلع والبكاء" التي انتابت الأطفال. للقصة جوانب ودلالات عديدة، فـ "بـطلها طفل يقود سيارة فارهة ويعرض حياته وحياة المارة للخطر، ويهين شرطيا طالبه باستظهار رخصته". مواضيع قد تهمك نهاية كل تلك المعلومات كانت كافية لجعل الموضوع يتصدر اهتمام المصريين، الذين حذروا أيضا مما وصفوها بـ"ظاهرة الإفلات من العقاب واستغلال النفوذ". فما التفاصيل؟ النيابة توضح في أواخر الأسبوع الماضي، انتشر مقطع للواقعة بكثافة عبر مواقع التواصل الاجتماعي، وسط غضب المصريين الذين طالبوا بمعاقبة الطفل ووالده. وتعود الواقعة إلى 26 أكتوبر / تشرين الثاني، عندما تقدم المارة بشكوى لشرطي مرور وأبلغوه بوجود شخص يقود سيارة برعونة، فاستوقف السيارة، ليكتشف أن السائق طفل يرافقه آخرون من نفس العمر. ويظهر الشرطي في الفيديو المتداول وهو يسأل الطفل عن رخصة القيادة، فصُدم بكيل من الشتائم تنهال عليه وبسخرية الأطفال منه وتوعدهم بإيذائه. ويتعذر على بي بي سي نشر الفيديو احتراما لخصوصية الأطفال. وبمجرد انتشار الفيديو عبر مواقع التواصل، أعلنت النيابة العامة في بيان نشرته على فيسبوك، فتح تحقيق في الواقعة واتخاذ الإجراءات اللازمة. لكن بيانا آخر يفيد بإخلاء سبيل الطفل بناء على طلب الأخصائي النفسي، قوبل بموجة استنكار، إذ طالب مغردون بـ"معاقبة الطفل وأبيه المستشار وتحميل الأخير المسؤولية كاملة حتى لا يتكرر الموقف ثانية". وكانت النيابة قد أشارت في بيانها الثاني إلى أن "السيارة التي كان يستقلها الطفل مملوكة لصديق والده، وقام باختلاس مفاتيح السيارة دون علم صاحبها". وحملت النيابة المسؤولية لصديق العائلة واكتفت بتغريمه 10 آلاف جنيه مصري أي ما يعادل 600 دولار أمريكي. ويبدو أن موجة الغضب قد دفعت السلطات إلى القبض على الطفل مرة أخرى. وأعلنت النيابة العامة في بيان نشرته مساء الاثنين، إيداع لـ "دار ملاحظة" لمدة أسبوع وطلب مذكرة من والده المستشار. كما قررت حبس أصدقائه أربعة أيام وعرضهم على الطب الشرعي لإجراء تحليل للتأكد من تعاطيهم للمخدرات. أزمة "مؤسسات" أم"أفراد"؟ وانتقد مغردون في بدء الأمر "تساهل السلطات في التعامل مع نجل مستشار انتهك القانون"، واحتفوا لاحقا بخبر إيقافه . في حين دعا المتحدث باسم نادي قضاة مصر إلى ضرورة الفصل بين الواقعة وصفة أبيه القضائية، مضيفا أنها "لن تعطل أو تتدخل في سير التحقيقات على عكس ما يروجه بعض المغرضين". كما حذر آخرون من تضخيم الموضوع وعملية "الشيطنة التي يقودها البعض ضد الطفل وأسرته" وفق تعبيرهم. وكان ممثلون وإعلاميون ومسؤولون قد عبروا عن غضبهم الشديد من الواقعة، التي اتفقوا على وصفها بـ "المشينة". ومن بين هؤلاء الممثل عبد الرحمن حسن والصحفي والبرلماني محمود بدر. واستدعى نشطاء قضايا لأطفال ومراهقين تم إيقافهم أو سجنهم لـ "أسباب بسيطة وأخرى تتعلق بآراء سياسية " قائلين إن "تلك القضايا كشفت ازدواجية الشرطة في التعامل مع أبناء الشعب الواحد" . كذلك طالب البعض بمحاسبة رجال شرطة المرور لـ"تهاونهم" في إيقاف الطفل، إذ غرد محمود سعيد قائلا" مقاطع عديدة تؤكد أن الطفل كرر فعلته أكثر من مرة، أمام أعين رجال الشرطة الذين اكتفوا بالتحديق به، لماذا يضع حدا للمشكلة؟ ثم كيف لأمه أن تتركه بدون رقابة؟" بينماأشاد آخرون بدور الشرطي الذي أصر على إيقاف الطفل وطلب منه رخصة السيارة، مطالبين بتكرميه. وما زاد من غضب البعض هو "نشر الطفل مقطعا عبر حسابه على انستغرام وهو يتباهى بسلطة والده، بعد انتهاء الاستجواب"، وفق ما قاله بعض المغردين. كما نشر البعض الآخر فيديوهات قالوا إنها تؤكد أن الطفل سبق أن قاد السيارة في مناسبات عديدة وأنه كان معتادا على إهانة رجال شرطة المرور" . وبحسب صحيفة "الدستور" المصرية فإن الشرطة ألقت القبض على الطفل وأربعة من أصدقائه بسبب إساءة استخدام وسائل التواصل الاجتماعي. وجددت تلك التفاصيل النقاش حول السبل التي يجب اتباعها لتقويم هذا السلوك. وفي الوقت الذي طالب فيه عمرو أديب بضم الطفل إلى "معسكر خاص يعلمه الانضباط واحترام الآخرين"، طالب آخرون بتحويله للأحداث، في حين عارض آخرون تلك الفكرة التي رأوا أن من شأنها هدم شخصية الطفل بدلا من إصلاحه. واستندت النيابة العامة في قرارها الأول، إلى رأي اختصاصي اجتماعي تابع للمجلس القومي للطفولة والأمومة. وذكر الاختصاصي في تقرير تداولته صحف محلية أن الطفل "مدلَل بشدة من والده". ونصح بـ"تسليمه لأهله والتعهد عليهم بتقويم سلوكه، وعقد جلسات دورية معه".'

In [ ]:
from transformers import AutoTokenizer
dir = '/content/my_awesome_billsum_model/checkpoint-9380'
tokenizer = AutoTokenizer.from_pretrained(dir)
inputs = tokenizer(text, return_tensors="pt").input_ids

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


In [ ]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(dir)
outputs = model.generate(inputs, max_new_tokens=100, do_sample=False)

In [ ]:
tokenizer.decode(outputs[0], skip_special_tokens=True)

'أثار مقطع فيديو لطفل مصري يقود سيارة فارهة، يثير جدلا واسعا عبر مواقع التواصل الاجتماعي في مصر، جدلا واسعا عبر مواقع التواصل الاجتماعي.'

In [ ]:
summary = []
for text in test_samples['text']:
  inputs = tokenizer(text, return_tensors="pt").input_ids
  outputs = model.generate(inputs, max_new_tokens=100, do_sample=False)
  summary.append(tokenizer.decode(outputs[0], skip_special_tokens=True))

Token indices sequence length is longer than the specified maximum sequence length for this model (1189 > 1024). Running this sequence through the model will result in indexing errors


In [ ]:
test_samples['Generated Summary'] = summary
test_samples.to_excel('Generated Summaries by AraT5.xlsx', index = False)
test_samples.head(2)

,text,summary,Generated Summary
0,واستحوذت القصة على نقاشات رواد مواقع التواصل ا...,أثار مقطع فيديو يظهر طفلا يقود سيارة ويعتدي لف...,أثار مقطع فيديو لطفل مصري يقود سيارة فارهة، يث...
1,وسيوفر البرنامج خمسة كيلوغرامات من الحبوب الرخ...,دشنت الحكومة الهندية برنامجا ضخما لتوفير الغذا...,أعلن مجلس الوزراء الهندي موافقته على برنامج دع...


In [ ]:
import pandas as pd
df = pd.read_excel('Generated Summaries by AraT5.xlsx')

In [ ]:
import re

# Optional: Arabic text normalization to improve BLEU alignment
def normalize_arabic(text):
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"[ًٌٍَُِّْ]", "", text)  # Remove short vowels (diacritics)
    text = re.sub(r"[^\w\s]", "", text)    # Remove punctuation (optional)
    return text.strip()

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import sacrebleu
import evaluate
from transformers import AutoTokenizer

# Initialize storage
bleu_scores = []
bleu_scores2 = []
rougeL_p = []
rougeL_r = []
rougeL_f = []
bertscore_p = []
bertscore_r = []
bertscore_f = []

# Initialize scorer
model_name = 'aubmindlab/bert-base-arabertv2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
rouge = rouge_scorer.RougeScorer(['rougeL'], tokenizer=tokenizer)

# Iterate through rows
for _, row in df.iterrows():
    raw_references = [row['summary']]  # One reference as list of strings
    raw_predictions = [row['Generated Summary']]  # One prediction

    # Normalize Arabic
    hyp = [normalize_arabic(text) for text in raw_predictions]
    ref = [[normalize_arabic(ref) for ref in raw_references]]  # Nested list for multiple refs per prediction

    # BLEU
    bleu = sacrebleu.corpus_bleu(hyp, ref)
    bleu_scores.append(bleu.score)

    bleu2 = evaluate.load("bleu")
    results = bleu2.compute(predictions= hyp, references= ref)
    bleu_scores2.append(results['bleu'])

    # ROUGE-L
    r_scores = rouge.score(row['summary'], row['Generated Summary'])
    rougeL_p.append(r_scores['rougeL'].precision)
    rougeL_r.append(r_scores['rougeL'].recall)
    rougeL_f.append(r_scores['rougeL'].fmeasure)

# BERTScore
P, R, F = bert_score(df['Generated Summary'].tolist(), df['summary'].tolist(), lang="ar", model_type="bert-base-multilingual-cased", verbose=False)
bertscore_p = P.tolist()
bertscore_r = R.tolist()
bertscore_f = F.tolist()

# Create summary DataFrame
metrics_df = pd.DataFrame({
    "BLEU1": bleu_scores,
    "BLEU2": bleu_scores2,
    "ROUGE_L_P": rougeL_p,
    "ROUGE_L_R": rougeL_r,
    "ROUGE_L_F": rougeL_f,
    "BERT_P": bertscore_p,
    "BERT_R": bertscore_r,
    "BERT_F": bertscore_f,
})

# Calculate min, max, mean
summary_stats = metrics_df.agg(['min', 'max', 'mean'])

print(summary_stats)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


          BLEU1     BLEU2  ROUGE_L_P  ROUGE_L_R  ROUGE_L_F    BERT_P  \
min    0.000000  0.000000   0.064516   0.060000   0.068182  0.656129   
max   44.476089  0.444761   0.789474   0.923077   0.779221  0.937043   
mean   8.971013  0.056827   0.340385   0.301111   0.311734  0.773840   

        BERT_R    BERT_F  
min   0.642207  0.651132  
max   0.948426  0.918719  
mean  0.752946  0.762822  
